# Probabilidade e distribuições em Python



In [11]:
#pip install scipy

In [ ]:
import math
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)   # semente/seed: as simulações dão sempre o mesmo resultado

---

A partir daqui, o `scipy.stats` faz o trabalho pesado. Toda distribuição tem os mesmos métodos, e vale a pena aprender esse padrão uma vez só:

| Método | O que devolve | Na aula |
|---|---|---|
| `.pmf(k)` | $P(X = k)$, para discretas | altura da barra |
| `.pdf(x)` | densidade, para contínuas | altura da curva (não é probabilidade!) |
| `.cdf(x)` | $P(X \le x)$ | a acumulada, a "tabela" |
| `.ppf(q)` | o valor $x$ com $P(X \le x) = q$ | o caminho inverso: percentil |
| `.mean()`, `.var()`, `.std()` | $E[X]$, $Var(X)$, desvio-padrão | os dois números que resumem a distribuição |
| `.rvs(size)` | amostras aleatórias | para simular |

## Distribuições discretas

### 1 Uniforme discreta: o dado

$$E[X] = \frac{a + b}{2} \qquad Var(X) = \frac{n^2 - 1}{12}$$

In [18]:
# randint(low, high) vai de low até high - 1: para um dado de 1 a 6, usamos (1, 7)
dado = stats.randint(1, 7)

print("P(X = 3) =", dado.pmf(3))
print("E[X]     =", dado.mean())
print("Var(X)   =", round(dado.var(), 2), " (fórmula: (36 - 1) / 12 =", round(35 / 12, 2), ")")

P(X = 3) = 0.16666666666666666
E[X]     = 3.5
Var(X)   = 2.92  (fórmula: (36 - 1) / 12 = 2.92 )


### 2 Binomial

$$P(X = k) = \binom{n}{k}\, p^k (1 - p)^{n - k} \qquad E[X] = np \qquad Var(X) = np(1 - p)$$

**Exemplo da aula.** A loja converte 25% dos visitantes. Numa amostra de 12 visitantes, qual a chance de exatamente 3 comprarem? E de pelo menos 1?


In [ ]:
X = stats.binom(n=12, p=0.25)

print(f"P(X = 3)      = {X.pmf(3):.4f}")
print(f"P(X >= 1)     = {1 - X.cdf(0)}") #tudo menos X=0
print(f"P(2 <= X <= 4)= {X.cdf(4) - X.cdf(1):.4f} ") 
# tudo até o 4 incluindo o 4 - tudo até o 2 incluindo o 2
# para incluir o 2, usamos cdf(1), que exclui o 1 mas não o 2


P(X = 3)      = 0.2581
P(X >= 1)     = 0.9683236479759216
P(2 <= X <= 4)= 0.6840 


### As outras discretas, no mesmo padrão

O guia rápido da aula vira uma tabela de funções. Os métodos são sempre os mesmos (`pmf`, `cdf`, `sf`, `mean`...):

| Pergunta | Distribuição | No `scipy.stats` |
|---|---|---|
| Sucessos em $n$ tentativas fixas? | Binomial | `binom(n, p)` |
| Eventos raros por intervalo? | Poisson | `poisson(mu)` |
| Tentativas até o 1º sucesso? | Geométrica | `geom(p)` |
| Fracassos até o $r$-ésimo sucesso? | Binomial negativa | `nbinom(r, p)` |
| Amostra sem reposição de um lote? | Hipergeométrica | `hypergeom(M, n, N)` |
| Todos igualmente prováveis? | Uniforme | `randint(a, b + 1)` |


---
## Distribuição Normal

A tabela Z da aula é a função `stats.norm.cdf`. E o caminho inverso (procurar a probabilidade no miolo da tabela e ler o $z$) é a `stats.norm.ppf`.

Uma vantagem: o `scipy` aceita a média e o desvio-padrão diretamente (`loc` = $\mu$ e `scale` = $\sigma$), então nem precisamos padronizar. Mas vamos fazer dos dois jeitos, para ver que dão o mesmo resultado.

### Escore-z e a tabela Z

$$z = \frac{x - \mu}{\sigma}$$

**Exemplo da aula.** A idade tem média 18 anos e desvio-padrão 3,57. Qual a chance de sortear uma pessoa com **mais de 20 anos**?

In [24]:
mu = 18
sigma = 3.57
x = 20

# Jeito 2: passa a média e o desvio direto, sem padronizar
idade = stats.norm(loc=mu, scale=sigma)
print(f"\nDireto: P(X > 20) = {1 - idade.cdf(20):.4f}")


Direto: P(X > 20) = 0.2877


A aula usou a tabela com $z = 0{,}56$ (arredondado) e chegou a 0,2877. O Python usa $z = 0{,}5602$ sem arredondar, por isso a diferença na quarta casa. O mesmo número também é um **percentil**: 20 anos está no percentil 71 da distribuição.



### O caminho inverso: do percentil para o valor

$$x = z \cdot \sigma + \mu$$

**Exemplo 1 da aula.** O IMC tem média 29 e desvio 6. Qual IMC marca o **percentil 90**?

In [ ]:
z_90 = stats.norm.ppf(0.90)           # o "procurar 0,900 no miolo da tabela"
imc_p90 = z_90 * 6 + 29
print(f"z do percentil 90 = {z_90:.4f}")
print(f"IMC do p90        = {imc_p90:.1f}")

# ou direto, sem padronizar
print(f"Direto: {stats.norm(loc=29, scale=6).ppf(0.90):.1f}")

z do percentil 90 = 1.2816
IMC do p90        = 36.7
Direto: 36.7


**Exemplo 2 da aula: achar $\mu$ e $\sigma$ a partir de duas probabilidades.** O tempo de uma tarefa é Normal. Sabe-se que $P(X \le 70) = 0{,}75$ e $P(X \le 50) = 0{,}25$.

Cada probabilidade vira um $z$ com `ppf`, e cada $z$ vira uma equação $x = \mu + z\sigma$. Duas equações, duas incógnitas:

In [25]:
z_75 = stats.norm.ppf(0.75)    # ≈ 0,6745
z_25 = stats.norm.ppf(0.25)    # ≈ -0,6745

# 70 = mu + z_75 * sigma
# 50 = mu + z_25 * sigma
# Subtraindo as equações: 20 = (z_75 - z_25) * sigma
sigma = (70 - 50) / (z_75 - z_25)
mu = 70 - z_75 * sigma
print(f"mu = {mu:.1f} minutos   sigma = {sigma:.2f} minutos")

# b) Porcentagem que precisa de mais de 85 minutos
tempo = stats.norm(loc=mu, scale=sigma)
print(f"P(X > 85) = {1-tempo.cdf(85):.4f}")

mu = 60.0 minutos   sigma = 14.83 minutos
P(X > 85) = 0.0459


### Gasto com alojamento e refeição

O gasto médio anual é de US$ 9.126, com desvio-padrão de US$ 1.500 na população. Numa amostra aleatória de 40 faculdades, qual a chance de a **média** ficar abaixo de US$ 9.400?

A única troca na fórmula do $z$ é o denominador, que passa a ser o erro-padrão:

$$z = \frac{\bar{x} - \mu}{\sigma / \sqrt{n}}$$

In [32]:
mu, sigma, n = 9126, 1500, 40

ep = sigma / np.sqrt(n)
z = (9400 - mu) / ep
print(f"Erro-padrão = {ep:.2f}")
print(f"z           = {z:.3f}")
print(f"P(média < 9.400) = {stats.norm.cdf(z):.4f}")

# Ou direto: a média amostral segue Normal(μ, σ/√n)
media_amostral = stats.norm(loc=mu, scale=ep)
print(f"Direto: {media_amostral.cdf(9400):.4f}")

Erro-padrão = 237.17
z           = 1.155
P(média < 9.400) = 0.8760
Direto: 0.8760


**Erro clássico:** usar $\sigma$ em vez de $\sigma/\sqrt{n}$. Isso responde outra pergunta, sobre **uma faculdade** sorteada, e não sobre a média de 40:

In [33]:
print(f"P(UMA faculdade < 9.400)      = {stats.norm(loc=mu, scale=sigma).cdf(9400):.4f}")
print(f"P(MÉDIA de 40 < 9.400)        = {media_amostral.cdf(9400):.4f}")

P(UMA faculdade < 9.400)      = 0.5725
P(MÉDIA de 40 < 9.400)        = 0.8760
